# Notebook 01 — Data Ingestion

Parse all Cricsheet IPL JSON files into a flat ball-by-ball DataFrame and save as parquet.

**Input:** `data/raw/ipl_json/*.json` (1,241 match files, 2008–2025 + partial 2026)  
**Output:** `data/processed/ball_by_ball.parquet`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.parse_cricsheet import parse_all

In [ ]:
# Parse all matches — takes ~1 second
df = parse_all()

In [ ]:
print(f'Total rows: {len(df):,}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Unique matches: {df["match_id"].nunique():,}')
df.dtypes

In [ ]:
# Matches per season
df.groupby('season')['match_id'].nunique().rename('matches').to_frame()

In [ ]:
# Stage breakdown
df['match_stage'].value_counts()

In [ ]:
# DLS-affected matches
print(f'DLS matches: {df[df["is_dls"]]["match_id"].nunique()}')

In [ ]:
# Quick sanity: top run-scorers all-time to validate batting_position and runs
league_only = df[df['match_stage'] == 'league']
(
    league_only.groupby('batter')
    .agg(runs=('runs_batter', 'sum'), balls=('runs_batter', 'count'))
    .sort_values('runs', ascending=False)
    .head(10)
)